In [47]:
from pathlib import Path
import json

# Find the root directory, no matter where we are. 
def find_project_root(marker="README.md"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root()


In [48]:

input_path = PROJECT_ROOT / "data/processed/feature_engineered_data.csv"
import pandas as pd
df = pd.read_csv(input_path)
    

In [49]:
input_path = PROJECT_ROOT / "data/progress_tracking/urls_with_descriptions.csv"
descriptions = pd.read_csv(input_path)
descriptions.rename(columns={'description': 'full_description'}, inplace=True)

In [50]:
df = df.merge(descriptions[['redirect_url', 'full_description']], on='redirect_url', how='left')

In [51]:
df.shape

(17101, 34)

In [52]:
df['full_description']  = df['full_description'].fillna('')

# Where df['full_desciption'] is of length < 10, replace it with df['description']
# df['full_description'] = df.apply(
 #   lambda row: row['description'] if len(str(row['full_description'])) < 10 else row['full_description'],
 #   axis=1
#)

df = df[df['full_description'].str.len() > 30]

In [ ]:
# 500 characters is the abridged descriptions. 
df['full_description'].str.len().value_counts()

full_description
500     8112
4420      40
3189      21
2569       9
4294       9
        ... 
9739       1
419        1
345        1
354        1
378        1
Name: count, Length: 4536, dtype: int64

In [36]:
df.shape

(17101, 34)

In [41]:
X_structured = df.drop(columns=['avg_salary', 'title', 'description'])
X_structured.columns

cat_vars = ['contract_type', 'contract_time', 'category.label', 'location.area_length', 
            # 'location_country', 
            'location_state', 'location_region_abridged', 'location_city_abridged', 'missing_long_lat']

num_vars = ['longitude', 'latitude']

In [42]:
df['location_state'].value_counts()

location_state
New South Wales                 5720
Victoria                        3378
Queensland                      3306
Western Australia               1451
South Australia                  570
Australian Capital Territory     509
Tasmania                         247
Northern Territory               220
Name: count, dtype: int64

In [43]:

def evaluate(y_true, y_pred):
    """
    Calculate regression metrics.
    Returns dict with rmse, mae, r2.
    """
    return {
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae': mean_absolute_error(y_true, y_pred),
        'r2': r2_score(y_true, y_pred)
    }

In [44]:
import joblib
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Refit on the full dataset — no train/val split.
# You've already evaluated; this is your production model.

print("Fitting production pipeline on full dataset...")

# --- Text: title (TF-IDF, no reduction) ---
title_tfidf = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2), min_df=3, sublinear_tf=True)
X_title_full = title_tfidf.fit_transform(df['title'].fillna(''))

# --- Text: description (TF-IDF + SVD) ---
desc_tfidf = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2), min_df=3, sublinear_tf=True)
desc_svd = TruncatedSVD(n_components=100, random_state=42)
X_desc_sparse = desc_tfidf.fit_transform(df['full_description'].fillna(''))
X_desc_full = desc_svd.fit_transform(X_desc_sparse)

# --- Structured features ---
ohe_prod = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
imputer_prod = SimpleImputer(strategy='median')
scaler_prod = StandardScaler()

cat_encoded_full = ohe_prod.fit_transform(df[cat_vars])
cont_imputed_full = imputer_prod.fit_transform(df[num_vars])
cont_scaled_full = scaler_prod.fit_transform(cont_imputed_full)

X_structured_full = np.hstack([cat_encoded_full, cont_scaled_full])

# --- Combine and fit final model ---
X_full = np.hstack([
    X_title_full.toarray(),
    X_desc_full,
    X_structured_full
])
y_full = df['avg_salary'].values

final_model = Ridge(alpha=1.0)
final_model.fit(X_full, y_full)
print(f"Final model fitted on {len(y_full)} samples.")

# --- Save bundle ---
pipeline_bundle = {
    'title_tfidf': title_tfidf,
    'desc_tfidf': desc_tfidf,
    'desc_svd': desc_svd,
    'ohe': ohe_prod,
    'imputer': imputer_prod,
    'scaler': scaler_prod,
    'model': final_model,
    'cat_vars': cat_vars,
    'num_vars': num_vars,
}

output_path = PROJECT_ROOT / 'models/salary_pipeline.joblib'
output_path.parent.mkdir(exist_ok=True)
joblib.dump(pipeline_bundle, output_path)
print(f"Pipeline saved to {output_path}")

Fitting production pipeline on full dataset...
Final model fitted on 17101 samples.
Pipeline saved to /home/seancm/code/salary_prediction_project/models/salary_pipeline.joblib


In [45]:
evaluate(final_model.predict(X_full), y_full)

{'rmse': np.float64(24746.265636251523),
 'mae': 16798.05397840392,
 'r2': 0.7102586302736713}

In [20]:
import joblib
b = joblib.load(PROJECT_ROOT / 'models/salary_pipeline.joblib')
for name, cats in zip(b['cat_vars'], b['ohe'].categories_):
    print(name, cats[:5])

contract_type ['contract' 'permanent' nan]
contract_time ['full_time' 'part_time' nan]
category.label ['Accounting & Finance Jobs' 'Admin Jobs' 'Charity & Voluntary Jobs'
 'Consultancy Jobs' 'Creative & Design Jobs']
location.area_length [1 2 3 4 5]
location_state ['Australian Capital Territory' 'New South Wales' 'Northern Territory'
 'Queensland' 'South Australia']
location_region_abridged ['Adelaide Region' 'Ballarat Region' 'Bathurst-Orange Region'
 'Bendigo Region' 'Brisbane Region']
location_city_abridged ['Adelaide' 'Boroondara Area' 'Brisbane' 'Caboolture Area' 'Cairns']
missing_long_lat [False  True]
